# Grammar-KT full-v1: frozen dataset and final evidence

This executable notebook reads the immutable public dataset and aggregate
experiment artifacts. It makes no model calls, does not mutate the dataset,
and never opens the learner-oracle trajectory. GrammarCells, generator K*, and
downstream K-hat hypotheses remain separate throughout.

In [1]:
import os

DATA_FOLDER = os.environ.get("GRAMMAR_KT_DATA_FOLDER", "data/grammar_kt_full_v1")
DATA_FOLDER

'data/grammar_kt_full_v1'

In [2]:
from collections import Counter
from pathlib import Path
import gzip
import hashlib
import json

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
DATASET = (ROOT / DATA_FOLDER).resolve()
assert DATASET.name == "grammar_kt_full_v1" and (DATASET / "manifest.json").is_file()

def load_json(relative):
    return json.loads((ROOT / relative).read_text(encoding="utf-8"))

def read_jsonl(path):
    opener = gzip.open if str(path).endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8") as stream:
        return [json.loads(line) for line in stream]

def show_table(rows, columns=None):
    frame = pd.DataFrame(rows)
    if columns:
        frame = frame.loc[:, columns]
    display(frame.reset_index(drop=True))
    return frame

SETUP = {"repo_root": str(ROOT), "dataset": str(DATASET), "read_only": True}
SETUP

{'repo_root': '/home/abdullah/grammar_kt_harness',
 'dataset': '/home/abdullah/grammar_kt_harness/data/grammar_kt_full_v1',
 'read_only': True}

## 1. Frozen release

In [3]:
manifest = json.loads((DATASET / "manifest.json").read_text(encoding="utf-8"))
release_rows = [
    {"object": key, "count": value}
    for key, value in manifest["scale"].items()
]
show_table(release_rows)

,object,count
0,source_relations,228
1,canonical_grammar_cells,75
2,generator_kcs,18
3,items,113
4,q_edges,269
5,learners,1000
6,interactions,283000


,object,count
0,source_relations,228
1,canonical_grammar_cells,75
2,generator_kcs,18
3,items,113
4,q_edges,269
5,learners,1000
6,interactions,283000


In [4]:
simulation = manifest["simulation"]["stream_summary"]
show_table([
    {"phase": phase, "events": count, "correct": simulation["correct_counts_by_phase"].get(phase)}
    for phase, count in simulation["phase_counts"].items()
])

,phase,events,correct
0,acquisition,170000,84438
1,probe,113000,65986


,phase,events,correct
0,acquisition,170000,84438
1,probe,113000,65986


## 2. Linguistic census and regimes

In [5]:
normalisation = json.loads((DATASET / "provenance/normalisation/full_audit.json").read_text(encoding="utf-8"))
counts = normalisation["final"]["results"]["counts"]
show_table([{"outcome": key, "count": value, "share": value / 1222} for key, value in counts.items()])

,outcome,count,share
0,complete,211,0.172668
1,partial,327,0.267594
2,unresolved,9,0.007365
3,out_of_scope,675,0.552373


,outcome,count,share
0,complete,211,0.172668
1,partial,327,0.267594
2,unresolved,9,0.007365
3,out_of_scope,675,0.552373


In [6]:
cells = read_jsonl(DATASET / "grammar/cells.jsonl")
regimes = read_jsonl(DATASET / "grammar/regime_assignments.jsonl")
regime_counts = Counter(row["grammar_regime"] for row in regimes)
show_table([{"grammar_regime": key, "cells": value} for key, value in sorted(regime_counts.items())])

,grammar_regime,cells
0,seen,54
1,unseen_combination,15
2,unseen_value,6


,grammar_regime,cells
0,seen,54
1,unseen_combination,15
2,unseen_value,6


## 3. Generator K* and measurement bank

In [7]:
kcs = read_jsonl(DATASET / "kcs.jsonl")
kc_rows = [{"kc_id": row["id"], "name": row["name"], "language_specific": row["language_specific"]} for row in kcs]
show_table(kc_rows)

,kc_id,name,language_specific
0,gkc_aspect_perfect,perfect construction,True
1,gkc_aspect_progressive,progressive construction,True
2,gkc_be_passive,canonical BE-passive,True
3,gkc_finite_past,past finite-form selection,True
4,gkc_finite_present,present finite-form selection,True
5,gkc_imperative,imperative clause formation,True
6,gkc_modal_can,central modal CAN,True
7,gkc_modal_could,central modal COULD,True
8,gkc_modal_may,central modal MAY,True
9,gkc_modal_might,central modal MIGHT,True


,kc_id,name,language_specific
0,gkc_aspect_perfect,perfect construction,True
1,gkc_aspect_progressive,progressive construction,True
2,gkc_be_passive,canonical BE-passive,True
3,gkc_finite_past,past finite-form selection,True
4,gkc_finite_present,present finite-form selection,True
5,gkc_imperative,imperative clause formation,True
6,gkc_modal_can,central modal CAN,True
7,gkc_modal_could,central modal COULD,True
8,gkc_modal_may,central modal MAY,True
9,gkc_modal_might,central modal MIGHT,True


In [8]:
items = read_jsonl(DATASET / "items/items.jsonl")
item_campaigns = Counter(
    row["generation_metadata"].get("campaign", "baseline_n3")
    for row in items
)
show_table([{"campaign": key, "selected_items": value} for key, value in sorted(item_campaigns.items())])

,campaign,selected_items
0,baseline_n3,89
1,cue_bounded_imperative_production_v1,4
2,explicit_construction_determinacy_intervention_v1,9
3,full_v1_validator_named_answer_packaging_corre...,1
4,unchanged_prompt_zero_coverage_rescue_v1,10


,campaign,selected_items
0,baseline_n3,89
1,cue_bounded_imperative_production_v1,4
2,explicit_construction_determinacy_intervention_v1,9
3,full_v1_validator_named_answer_packaging_corre...,1
4,unchanged_prompt_zero_coverage_rescue_v1,10


In [9]:
measurement = json.loads((DATASET / "provenance/measurement/audit.json").read_text(encoding="utf-8"))
show_table([{"diagnostic": key, "value": value} for key, value in measurement["counts"].items()])

,diagnostic,value
0,canonical_cells,75
1,measured_cells,75
2,items,113
3,generator_kcs,18
4,q_edges,269
5,q_density,0.132252
6,q_rank,18
7,full_column_rank,True
8,distinct_canonical_cell_activation_rows,75
9,kc_pairs,153


,diagnostic,value
0,canonical_cells,75
1,measured_cells,75
2,items,113
3,generator_kcs,18
4,q_edges,269
5,q_density,0.132252
6,q_rank,18
7,full_column_rank,True
8,distinct_canonical_cell_activation_rows,75
9,kc_pairs,153


## 4. RQ2 — controlled KC/Q misspecification

In [10]:
rq2 = load_json("reports/full_v1_artifacts/rq2_misspecification_v1/results.json")
rq2_order = ["all_merged", "coarse_linguistic_families", "true_kstar", "structural_split2", "structural_split4", "exact_cell"]
rq2_rows = []
reference_loss = rq2["metrics_by_representation"]["true_kstar"]["probe_metrics"]["all_probe"]["log_loss"]
for representation in rq2_order:
    record = rq2["metrics_by_representation"][representation]
    metric = record["probe_metrics"]["all_probe"]
    rq2_rows.append({
        "representation": representation,
        "KCs": record["model_audit"]["hypothesis_kcs"],
        "log_loss": metric["log_loss"],
        "delta_vs_K*": metric["log_loss"] - reference_loss,
        "brier": metric["brier_score"],
    })
show_table(rq2_rows)

,representation,KCs,log_loss,delta_vs_K*,brier
0,all_merged,1,0.680852,0.010225,0.243873
1,coarse_linguistic_families,6,0.678759,0.008132,0.242857
2,true_kstar,18,0.670627,0.000000,0.238925
3,structural_split2,35,0.673792,0.003165,0.240471
4,structural_split4,66,0.676495,0.005868,0.241794
5,exact_cell,75,0.685666,0.015039,0.246348


,representation,KCs,log_loss,delta_vs_K*,brier
0,all_merged,1,0.680852,0.010225,0.243873
1,coarse_linguistic_families,6,0.678759,0.008132,0.242857
2,true_kstar,18,0.670627,0.000000,0.238925
3,structural_split2,35,0.673792,0.003165,0.240471
4,structural_split4,66,0.676495,0.005868,0.241794
5,exact_cell,75,0.685666,0.015039,0.246348


## 5. RQ3 — observable-only discovery

In [11]:
rq3 = load_json("experiments/full_v1/rq3_kc_discovery_v1/final_evaluation.json")
rq3_ids = ["compositional_operations", "atomic_features", "coarse_operations", "fine_exact_cells", "hash_distractor_negative_control"]
rq3_rows = []
for policy in rq3_ids:
    recovery = rq3["structural_recovery"][policy]["all"]
    prediction = rq3["predictive_probe_evaluation"][policy]["all_probes"]
    rq3_rows.append({
        "policy": policy,
        "exact_true_KCs": recovery["characterisation"]["exact_true_kcs_recovered"],
        "activation_Jaccard": recovery["optimal_matching"]["mean_activation_jaccard_padded"],
        "aligned_Q_F1": recovery["aligned_q_edges"]["f1"],
        "probe_log_loss": prediction["log_loss"],
    })
show_table(rq3_rows)

,policy,exact_true_KCs,activation_Jaccard,aligned_Q_F1,probe_log_loss
0,compositional_operations,18,1.000000,1.000000,0.669606
1,atomic_features,16,0.970854,0.965385,0.669979
2,coarse_operations,5,0.371913,0.750929,0.675470
3,fine_exact_cells,1,0.084184,0.186969,0.676262
4,hash_distractor_negative_control,0,0.202342,0.359259,0.682843


,policy,exact_true_KCs,activation_Jaccard,aligned_Q_F1,probe_log_loss
0,compositional_operations,18,1.000000,1.000000,0.669606
1,atomic_features,16,0.970854,0.965385,0.669979
2,coarse_operations,5,0.371913,0.750929,0.675470
3,fine_exact_cells,1,0.084184,0.186969,0.676262
4,hash_distractor_negative_control,0,0.202342,0.359259,0.682843


## 6. RQ4 — linguistic generalisation

In [12]:
rq4 = load_json("experiments/full_v1/rq4_generalisation_v1/results.json")
rq4_rows = []
for representation, record in rq4["baseline_generalisation"].items():
    for regime in ("seen", "unseen_combination", "unseen_value"):
        metric = record["by_grammar_regime"][regime]["event_weighted"]
        rq4_rows.append({"representation": representation, "regime": regime, "log_loss": metric["log_loss"], "brier": metric["brier_score"]})
show_table(rq4_rows)

,representation,regime,log_loss,brier
0,true_kstar_compositional_ceiling,seen,0.669161,0.238209
1,true_kstar_compositional_ceiling,unseen_combination,0.672036,0.239621
2,true_kstar_compositional_ceiling,unseen_value,0.681181,0.244059
3,rq3_atomic,seen,0.669161,0.238209
4,rq3_atomic,unseen_combination,0.672036,0.239621
5,rq3_atomic,unseen_value,0.677945,0.242359
6,family_union_coarse,seen,0.678101,0.242533
7,family_union_coarse,unseen_combination,0.681221,0.244068
8,family_union_coarse,unseen_value,0.679430,0.243188
9,structural_split2,seen,0.671395,0.239302


,representation,regime,log_loss,brier
0,true_kstar_compositional_ceiling,seen,0.669161,0.238209
1,true_kstar_compositional_ceiling,unseen_combination,0.672036,0.239621
2,true_kstar_compositional_ceiling,unseen_value,0.681181,0.244059
3,rq3_atomic,seen,0.669161,0.238209
4,rq3_atomic,unseen_combination,0.672036,0.239621
5,rq3_atomic,unseen_value,0.677945,0.242359
6,family_union_coarse,seen,0.678101,0.242533
7,family_union_coarse,unseen_combination,0.681221,0.244068
8,family_union_coarse,unseen_value,0.679430,0.243188
9,structural_split2,seen,0.671395,0.239302


## 7. Oracle-only state evaluation

In [13]:
mastery = load_json("reports/full_v1_artifacts/mastery_recovery_v1/results.json")
mastery_rows = []
for representation, record in mastery["metrics_by_representation"].items():
    metric = record["metrics_by_regime"]["all_probe"]
    mastery_rows.append({
        "representation": representation,
        "RMSE": metric["rmse"],
        "MAE": metric["mae"],
        "correlation": metric["pearson_correlation"],
        "bias": metric["calibration"]["mean_estimate_minus_oracle"],
    })
show_table(mastery_rows)

,representation,RMSE,MAE,correlation,bias
0,true_kstar,0.123738,0.100560,0.569746,-0.025206
1,coarse_linguistic_families,0.146300,0.120514,0.267969,-0.036060
2,structural_split2,0.132752,0.107583,0.465799,-0.025716
3,structural_split4,0.140428,0.114151,0.383038,-0.031960
4,exact_cell,0.163828,0.133419,0.291646,-0.070962


,representation,RMSE,MAE,correlation,bias
0,true_kstar,0.123738,0.100560,0.569746,-0.025206
1,coarse_linguistic_families,0.146300,0.120514,0.267969,-0.036060
2,structural_split2,0.132752,0.107583,0.465799,-0.025716
3,structural_split4,0.140428,0.114151,0.383038,-0.031960
4,exact_cell,0.163828,0.133419,0.291646,-0.070962


## 8. Compact simulator robustness

In [14]:
robustness = load_json("experiments/full_v1/simulator_robustness_v1/results.json")
robust_rows = []
for row in robustness["summary"]["candidate_minus_kstar_by_condition_across_seeds"]:
    if row["analysis_role"] != "primary":
        continue
    delta = row["delta_log_loss"]
    robust_rows.append({
        "condition": row["condition_id"],
        "candidate": row["candidate_representation"],
        "mean_delta_LL": delta["mean"],
        "minimum": delta["minimum"],
        "maximum": delta["maximum"],
    })
show_table(robust_rows)

,condition,candidate,mean_delta_LL,minimum,maximum
0,baseline_minimum_g10_s10,coarse_linguistic_families,0.007924,0.007568,0.008306
1,baseline_minimum_g10_s10,structural_split2,0.003241,0.003001,0.003575
2,noise_g00_s00,coarse_linguistic_families,0.016363,0.016032,0.016957
3,noise_g00_s00,structural_split2,0.005969,0.005573,0.006636
4,noise_g20_s10,coarse_linguistic_families,0.005259,0.004938,0.005684
5,noise_g20_s10,structural_split2,0.002222,0.001971,0.002457
6,noise_g10_s20,coarse_linguistic_families,0.005310,0.005110,0.005692
7,noise_g10_s20,structural_split2,0.002283,0.002159,0.002526
8,noise_g20_s20,coarse_linguistic_families,0.003082,0.002904,0.003436
9,noise_g20_s20,structural_split2,0.001506,0.001372,0.001604


,condition,candidate,mean_delta_LL,minimum,maximum
0,baseline_minimum_g10_s10,coarse_linguistic_families,0.007924,0.007568,0.008306
1,baseline_minimum_g10_s10,structural_split2,0.003241,0.003001,0.003575
2,noise_g00_s00,coarse_linguistic_families,0.016363,0.016032,0.016957
3,noise_g00_s00,structural_split2,0.005969,0.005573,0.006636
4,noise_g20_s10,coarse_linguistic_families,0.005259,0.004938,0.005684
5,noise_g20_s10,structural_split2,0.002222,0.001971,0.002457
6,noise_g10_s20,coarse_linguistic_families,0.005310,0.005110,0.005692
7,noise_g10_s20,structural_split2,0.002283,0.002159,0.002526
8,noise_g20_s20,coarse_linguistic_families,0.003082,0.002904,0.003436
9,noise_g20_s20,structural_split2,0.001506,0.001372,0.001604


## 9. Collection-design controls

In [15]:
collection_path = ROOT / "experiments/full_v1/collection_design_v1/results.json"
assert collection_path.is_file(), "Frozen collection-design result is required"
collection = json.loads(collection_path.read_text(encoding="utf-8"))
boundary = collection["boundary_audit"]
assert boundary["baseline_immutable"] is True
assert boundary["probe_outcomes_used_for_selection"] is False
assert boundary["oracle_state_exposed_to_predictor"] is False
show_table([
    {"learners": int(learners), **record}
    for learners, record in collection["A_learner_count_stability"]["selection_frequency"].items()
])

,learners,replicates,kstar_predictive_selection_frequency,kstar_penalized_selection_frequency,predictive_winner_counts,penalized_winner_counts
0,60,5,1,0.2,{'true_kstar': 5},"{'family_union_coarse': 4, 'true_kstar': 1}"
1,120,5,1,0.0,{'true_kstar': 5},{'family_union_coarse': 5}
2,240,5,1,0.0,{'true_kstar': 5},{'family_union_coarse': 5}
3,500,5,1,0.4,{'true_kstar': 5},"{'family_union_coarse': 3, 'true_kstar': 2}"
4,1000,1,1,0.0,{'true_kstar': 1},{'family_union_coarse': 1}


,learners,replicates,kstar_predictive_selection_frequency,kstar_penalized_selection_frequency,predictive_winner_counts,penalized_winner_counts
0,60,5,1,0.2,{'true_kstar': 5},"{'family_union_coarse': 4, 'true_kstar': 1}"
1,120,5,1,0.0,{'true_kstar': 5},{'family_union_coarse': 5}
2,240,5,1,0.0,{'true_kstar': 5},{'family_union_coarse': 5}
3,500,5,1,0.4,{'true_kstar': 5},"{'family_union_coarse': 3, 'true_kstar': 2}"
4,1000,1,1,0.0,{'true_kstar': 1},{'family_union_coarse': 1}


In [16]:
opportunity_rows = []
for target, representations in collection["B_opportunities_per_learner"]["summary"].items():
    for representation, regimes_at_target in representations.items():
        opportunity_rows.append({
            "target_opportunities": int(target),
            "representation": representation,
            "all_probe_log_loss": regimes_at_target["all_probe"]["mean_log_loss"],
            "combination_log_loss": regimes_at_target["unseen_combination"]["mean_log_loss"],
        })
show_table(opportunity_rows)

,target_opportunities,representation,all_probe_log_loss,combination_log_loss
0,6,true_kstar,0.681757,0.684428
1,6,family_union_coarse,0.687194,0.690072
2,6,structural_split2,0.684468,0.691236
3,6,exact_cell,0.689280,0.703218
4,12,true_kstar,0.672104,0.673250
5,12,family_union_coarse,0.679989,0.681718
6,12,structural_split2,0.675131,0.681536
7,12,exact_cell,0.686405,0.709853
8,24,true_kstar,0.636837,0.638261
9,24,family_union_coarse,0.646908,0.649012


,target_opportunities,representation,all_probe_log_loss,combination_log_loss
0,6,true_kstar,0.681757,0.684428
1,6,family_union_coarse,0.687194,0.690072
2,6,structural_split2,0.684468,0.691236
3,6,exact_cell,0.689280,0.703218
4,12,true_kstar,0.672104,0.673250
5,12,family_union_coarse,0.679989,0.681718
6,12,structural_split2,0.675131,0.681536
7,12,exact_cell,0.686405,0.709853
8,24,true_kstar,0.636837,0.638261
9,24,family_union_coarse,0.646908,0.649012


In [17]:
item_support = collection["C_items_per_kc"]
show_table([
    {
        "bank": bank,
        "items": record["items"],
        "unique_Q_rows": record["unique_q_rows"],
        "Q_rank": record["rank"],
        "minimum_items_per_KC": record["item_support_per_kc"]["minimum"],
        "median_items_per_KC": record["item_support_per_kc"]["median"],
    }
    for bank, record in item_support.items()
    if bank.startswith("max_")
])

,bank,items,unique_Q_rows,Q_rank,minimum_items_per_KC,median_items_per_KC
0,max_one_per_cell,75,75,18,1,5.0
1,max_two_per_cell,113,75,18,2,7.5


,bank,items,unique_Q_rows,Q_rank,minimum_items_per_KC,median_items_per_KC
0,max_one_per_cell,75,75,18,1,5.0
1,max_two_per_cell,113,75,18,2,7.5


In [18]:
anchor_rows = []
for world, designs in collection["D_anchor_identifiability"]["summary"].items():
    for design, learner_results in designs.items():
        for learners, comparisons in learner_results.items():
            for comparison, metric in comparisons.items():
                anchor_rows.append({
                    "world": world,
                    "design": design,
                    "learners": int(learners),
                    "comparison": comparison,
                    "mean_delta_log_loss_vs_true": metric["mean_delta_log_loss_vs_true"],
                    "seed_minimum": metric["seed_range"][0],
                    "seed_maximum": metric["seed_range"][1],
                })
show_table(anchor_rows)

,world,design,learners,comparison,mean_delta_log_loss_vs_true,seed_minimum,seed_maximum
0,factorized_ab,all_ab_no_anchors,100,union_merge,0.000000,0.000000,0.000000
1,factorized_ab,all_ab_no_anchors,100,spurious_intersection,0.000000,0.000000,0.000000
2,factorized_ab,all_ab_no_anchors,300,union_merge,0.000000,0.000000,0.000000
3,factorized_ab,all_ab_no_anchors,300,spurious_intersection,0.000000,0.000000,0.000000
4,factorized_ab,all_ab_no_anchors,1000,union_merge,0.000000,0.000000,0.000000
5,factorized_ab,all_ab_no_anchors,1000,spurious_intersection,0.000000,0.000000,0.000000
6,factorized_ab,sparse_anchors,100,union_merge,0.004348,0.004012,0.004684
7,factorized_ab,sparse_anchors,100,spurious_intersection,-0.000007,-0.000193,0.000123
8,factorized_ab,sparse_anchors,300,union_merge,0.007872,0.005337,0.011603
9,factorized_ab,sparse_anchors,300,spurious_intersection,0.000038,-0.000008,0.000128


,world,design,learners,comparison,mean_delta_log_loss_vs_true,seed_minimum,seed_maximum
0,factorized_ab,all_ab_no_anchors,100,union_merge,0.000000,0.000000,0.000000
1,factorized_ab,all_ab_no_anchors,100,spurious_intersection,0.000000,0.000000,0.000000
2,factorized_ab,all_ab_no_anchors,300,union_merge,0.000000,0.000000,0.000000
3,factorized_ab,all_ab_no_anchors,300,spurious_intersection,0.000000,0.000000,0.000000
4,factorized_ab,all_ab_no_anchors,1000,union_merge,0.000000,0.000000,0.000000
5,factorized_ab,all_ab_no_anchors,1000,spurious_intersection,0.000000,0.000000,0.000000
6,factorized_ab,sparse_anchors,100,union_merge,0.004348,0.004012,0.004684
7,factorized_ab,sparse_anchors,100,spurious_intersection,-0.000007,-0.000193,0.000123
8,factorized_ab,sparse_anchors,300,union_merge,0.007872,0.005337,0.011603
9,factorized_ab,sparse_anchors,300,spurious_intersection,0.000038,-0.000008,0.000128


## 10. Integrity and RQ ledger

In [19]:
hash_rows = []
for relative in ["manifest.json", "q_matrix.csv", "interactions.jsonl.gz"]:
    path = DATASET / relative
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    expected = manifest["artifact_inventory"].get(relative, {}).get("sha256")
    hash_rows.append({
        "artifact": relative,
        "sha256": observed,
        "bytes": path.stat().st_size,
        "matches_manifest": observed == expected if expected else "self-manifest",
    })
show_table(hash_rows)

,artifact,sha256,bytes,matches_manifest
0,manifest.json,322128843f8e7e6547a99efcecc6836fcd581f314fb495...,26456,self-manifest
1,q_matrix.csv,b6df582478f05976ceb200da6edc2b31fb305da64498e5...,8404,True
2,interactions.jsonl.gz,9272ca86a647e3b13c9ce52b5381dde215f7ef448e4a19...,2158088,True


,artifact,sha256,bytes,matches_manifest
0,manifest.json,322128843f8e7e6547a99efcecc6836fcd581f314fb495...,26456,self-manifest
1,q_matrix.csv,b6df582478f05976ceb200da6edc2b31fb305da64498e5...,8404,True
2,interactions.jsonl.gz,9272ca86a647e3b13c9ce52b5381dde215f7ef448e4a19...,2158088,True


In [20]:
FINAL_DATASET_SUMMARY = {
    "scientific_distinction": "GrammarCell != generator K* != discovered K_hat",
    "dataset_status": manifest["status"],
    "cells": manifest["scale"]["canonical_grammar_cells"],
    "generator_kcs": manifest["scale"]["generator_kcs"],
    "items": manifest["scale"]["items"],
    "events": manifest["scale"]["interactions"],
    "RQ1": "supported within declared scope",
    "RQ2": "supported for frozen perturbations; item-difficulty caveat",
    "RQ3": "unique recovery rejected; equivalence-class recovery supported",
    "RQ4": "recombination supported; unseen-value ontology choice inconclusive",
    "oracle_trajectory_opened": False,
}
FINAL_DATASET_SUMMARY

{'scientific_distinction': 'GrammarCell != generator K* != discovered K_hat',
 'dataset_status': 'FROZEN_BASELINE_COMPLETE',
 'cells': 75,
 'generator_kcs': 18,
 'items': 113,
 'events': 283000,
 'RQ1': 'supported within declared scope',
 'RQ2': 'supported for frozen perturbations; item-difficulty caveat',
 'RQ3': 'unique recovery rejected; equivalence-class recovery supported',
 'RQ4': 'recombination supported; unseen-value ontology choice inconclusive',
 'oracle_trajectory_opened': False}

## Interpretation boundary

The notebook demonstrates retained artifacts; it does not regenerate LLM
annotations, fit policies on holdout outcomes, or expose oracle trajectories.
K*, simulator parameters, and sample-size results are controlled synthetic
truths—not estimates of human cognition. Automatic item judgments are not human
pedagogical validation, and the alternate-schema contract is not cross-lingual
empirical evidence.